# 07 — Support Vector Machines and Kernel Methods

This notebook implements a linear SVM from scratch using subgradient descent and explores kernel functions.

In [ ]:
import numpy as np

## 1. Data

In [ ]:
rng = np.random.default_rng(42)

n = 220
class_neg = rng.multivariate_normal([-1.8, -0.8], [[0.35, 0.08], [0.08, 0.35]], size=n // 2)
class_pos = rng.multivariate_normal([1.7, 0.9], [[0.35, -0.05], [-0.05, 0.35]], size=n // 2)

X = np.vstack([class_neg, class_pos])
y = np.array([-1] * (n // 2) + [1] * (n // 2))

X.shape, y.shape

## 2. Split and Scale

In [ ]:
def train_test_split_numpy(X, y, test_size=0.25, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y)
    idx = rng.permutation(n)
    test_n = int(n * test_size)
    return X[idx[test_n:]], X[idx[:test_n]], y[idx[test_n:]], y[idx[:test_n]]


def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std = np.where(std == 0, 1, std)
    return (X_train - mean) / std, (X_test - mean) / std

X_train, X_test, y_train, y_test = train_test_split_numpy(X, y)
X_train_scaled, X_test_scaled = standardize_train_test(X_train, X_test)

## 3. Hinge Loss

$$
\ell_i = \max(0, 1-y_i(w^Tx_i+b))
$$

In [ ]:
def hinge_loss(y, scores):
    margins = y * scores
    return np.mean(np.maximum(0, 1 - margins))

## 4. Linear SVM from Scratch

In [ ]:
def train_linear_svm(X, y, C=1.0, lr=0.01, epochs=2500):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    losses = []

    for epoch in range(epochs):
        scores = X @ w + b
        margins = y * scores
        hinge = np.maximum(0, 1 - margins)
        objective = 0.5 * np.dot(w, w) + C * np.mean(hinge)
        losses.append(objective)

        violating = margins < 1
        if np.any(violating):
            grad_w = w - C * np.mean(y[violating, None] * X[violating], axis=0)
            grad_b = -C * np.mean(y[violating])
        else:
            grad_w = w
            grad_b = 0.0

        step = lr / (1 + 0.0005 * epoch)
        w -= step * grad_w
        b -= step * grad_b

    return w, b, np.array(losses)

w, b, losses = train_linear_svm(X_train_scaled, y_train, C=2.0, lr=0.04, epochs=3000)

w, b, losses[0], losses[-1]

## 5. Predictions and Metrics

In [ ]:
def predict_linear_svm(X, w, b):
    scores = X @ w + b
    return np.where(scores >= 0, 1, -1)

pred = predict_linear_svm(X_test_scaled, w, b)
accuracy = np.mean(pred == y_test)
accuracy

## 6. Support Vector Intuition

Support vectors are points close to or inside the margin:

$$
y_i(w^Tx_i+b) \leq 1
$$

In [ ]:
train_scores = X_train_scaled @ w + b
train_margins = y_train * train_scores
support_mask = train_margins <= 1.05
np.sum(support_mask), len(y_train)

## 7. Kernel Functions

In [ ]:
def linear_kernel(X, Z):
    return X @ Z.T


def polynomial_kernel(X, Z, degree=3, coef0=1.0):
    return (X @ Z.T + coef0) ** degree


def rbf_kernel(X, Z, gamma=1.0):
    X_norm = np.sum(X ** 2, axis=1)[:, None]
    Z_norm = np.sum(Z ** 2, axis=1)[None, :]
    sq_dist = X_norm + Z_norm - 2 * X @ Z.T
    return np.exp(-gamma * sq_dist)

K = rbf_kernel(X_train_scaled[:5], X_train_scaled[:5], gamma=0.7)
K

## Reflection

SVM is geometry plus optimization plus similarity: maximize the margin, identify support vectors, and use kernels to make nonlinear boundaries possible.